# 01. TabPFN-3 한눈에 보기와 셀 encoder

TabPFN-3(Prior Labs, *TabPFN-3: Technical Report*, arXiv 2605.13986, 2026)는 v2와 2.5가 쓰던 셀 단위 2방향 attention을 버리고, 표를 세 단계로 나눠 읽는 구조로 바꿨습니다. 먼저 열마다 값의 분포를 요약하고(열 단계), 행마다 셀들을 벡터 하나로 압축한 뒤(행 단계), 그 행 벡터들로 in-context learning을 합니다(행 ICL). TabICL(Qu et al., 2025)이 먼저 쓴 뼈대이고, 최근 나온 tabular foundation model(TFM) 여럿(TabICLv2, Causilo, TabAbax 등)도 이 형태입니다.

이 폴더의 두 노트북은 2.5 노트북의 과일 표를 세 단계에 하나씩 통과시키며, 각 단계가 무엇을 받아 무엇을 내는지 실제 값으로 따라갑니다. 이 노트북은 구조와 파라미터 수, 왜 바꿨는지(계산이 어디에 드는지), 그리고 세 단계에 들어가기 전의 셀 encoder를 봅니다. 열 단계부터 출력까지는 [02 열 → 행 → 행 ICL 세 단계](02_column_row_icl_stages.ipynb)에서 이어갑니다. 코드는 `src/minipfn/models/column_row_transformer.py`(`ColumnRowTransformer`)이고, 3과 3.5는 같은 클래스에 설정만 다릅니다(`src/minipfn/recipes/tabpfn_v3.py`의 `ARCHITECTURES`). 3.5가 바꾼 것과 분류·회귀를 모델 하나가 하는 multitask 구조는 [06_tabpfn_v3_5/01 TabPFN-3.5가 바꾼 것과 multitask 구조](../06_tabpfn_v3_5/01_changes_and_multitask.ipynb)에서 다룹니다. 비교 대상인 v2·2.5 구조는 [03_tabpfn_v2/01 셀 단위 표현과 2방향 attention](../03_tabpfn_v2/01_cell_tokens_and_two_way_attention.ipynb)과 [04_tabpfn_v2_5/01](../04_tabpfn_v2_5/01_layers_and_feature_groups.ipynb)·[02 TabPFN-2.5의 구조 변경](../04_tabpfn_v2_5/02_thinking_rows_and_regression_encoder.ipynb)에서 다뤘습니다. v1부터 3.5까지의 차이는 [노트북 로드맵의 버전 비교 표](../README.md#tabpfn-버전-비교)에 모았고, 이 폴더는 그 표의 2.5 → 3 칸을 따라갑니다.

다루는 흐름:

1. 한눈에 보기 — 세 단계와 파라미터 수
2. 왜 바꿨나 — 계산이 어디에 드는가
3. 셀 encoder — 셀 3개 묶기와 정답 embedding
4. 정리

## `minipfn` 불러오기

다른 노트북과 같은 import shim입니다. 이 노트북은 `notebooks/05_tabpfn_v3/` 아래에 있으므로 저장소 루트는 두 단계 위입니다.

In [1]:
try:
    import minipfn
except ImportError:
    import sys
    from pathlib import Path

    sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
    import minipfn

In [2]:
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn

from minipfn import set_seed
from minipfn.models import randomize_zero_weights
from minipfn.models.column_row_transformer import (
    ColumnRowTransformer,
    ColumnRowTransformerConfig,
    TargetEncoder,
    group_offsets,
)
from minipfn.recipes import tabpfn_v2
from minipfn.recipes.tabpfn_v3 import ARCHITECTURES, paper

set_seed(0)  # 재현성을 위해 난수 시드를 고정한다.

## 1. 한눈에 보기 — 세 단계와 파라미터 수

`ARCHITECTURES`에는 구조마다, task마다 모델 설정이 들어 있습니다. 3은 분류와 회귀가 같은 구조이고 출력층만 다릅니다. 3.5의 설정은 [06 노트북](../06_tabpfn_v3_5/01_changes_and_multitask.ipynb)에서 봅니다.

In [3]:
rows = {f"v3 {task}": settings for task, settings in ARCHITECTURES["v3"].items()}
display(pd.DataFrame(rows).T.drop(columns="model_type"))

,emsize,nlayers,nhead,features_per_group,n_inducing,n_cls,qk_norm,encoder,target_norm,head_mlp
v3 classification,128,24,8,3,128,4,False,linear,False,False
v3 regression,128,24,8,3,128,4,False,linear,False,False


`emsize`(E = 128)는 셀·열·행 단계의 폭이고, `n_cls`는 행 하나를 압축할 CLS 토큰 수입니다. 행 ICL의 폭은 `n_cls × E`라서 3은 512입니다. `nlayers`와 `nhead`는 행 ICL의 층 수와 head 수입니다. 나머지 네 열(`qk_norm`, `encoder`, `target_norm`, `head_mlp`)은 3.5에서 바뀌는 것이라 [06 노트북](../06_tabpfn_v3_5/01_changes_and_multitask.ipynb)에서 봅니다.

실제 크기의 모델을 만들어 파라미터를 단계별로 셉니다. 학습 설정 `paper(task, arch)`의 `build_model()`은 학습에 쓰는 모델과 같은 것을 만듭니다. 모델의 `num_parameters_by_stage()`가 맨 앞 모듈 이름(`column_blocks`, `icl_layers` 등)을 단계로 묶어 셉니다.

In [4]:
# 모델이 단계마다 센 파라미터 수(`num_parameters_by_stage`)에 붙일 이름
STAGE_LABEL = {
    "cell_encoder": "셀 encoder",
    "column_stage": "열 단계",
    "row_stage": "행 단계",
    "row_icl": "행 ICL",
    "output": "출력",
}


def stage_counts(model):
    counts = {STAGE_LABEL[k]: v for k, v in model.num_parameters_by_stage().items()}
    counts["합계"] = model.num_parameters()
    return counts


counts = {}
for task in ["classification", "regression"]:
    model = paper(task, "v3").build_model()
    counts[f"v3 {task}"] = stage_counts(model)
    del model

order = ["셀 encoder", "열 단계", "행 단계", "행 ICL", "출력", "합계"]
param_table = pd.DataFrame(counts).reindex(order).fillna(0).astype(int)
display(param_table.style.format("{:,}"))

v25 = tabpfn_v2.with_architecture(tabpfn_v2.paper("classification"), "v2.5").build_model()
print(f"비교: TabPFN-2.5 분류 모델 {v25.num_parameters():,}")

,v3 classification,v3 regression
셀 encoder,"2,176","1,152"
열 단계,"869,616","869,616"
행 단계,"394,624","394,624"
행 ICL,"51,363,328","51,359,232"
출력,"427,392","5,650,312"
합계,"53,057,136","58,274,936"


비교: TabPFN-2.5 분류 모델 14,335,306


파라미터는 거의 다 행 ICL(24층)에 있습니다. 3 분류 모델에서 셀 encoder, 열 단계, 행 단계를 다 합쳐도 127만 개이고, 행 ICL이 5,136만 개입니다. 회귀 모델은 출력층이 bar distribution 구간 5,000개의 logit을 내느라 520만 개쯤 더 큽니다.

TabPFN-2.5 분류 모델(1,434만 개)과 비교하면 3은 파라미터가 4배 가까이 많습니다. 그런데 다음 절에서 보듯이 계산은 오히려 적습니다.

## 2. 왜 바꿨나 — 계산이 어디에 드는가

v2·2.5는 셀(묶음) 하나하나가 토큰이고, 층마다 두 방향으로 attention합니다. 행 $R$개, 학습 행 $N$개, 토큰 열 $C$개라고 하면 층 하나에서 다음을 계산합니다.

- feature attention: 행마다 토큰 $C$개가 서로를 봅니다. $R \cdot C^2$
- sample attention: 열마다 행 $R$개가 학습 행 $N$개를 봅니다. $C \cdot R \cdot N$

둘째 항이 문제입니다. 행 수와 열 수가 곱해져서 층마다 다시 계산되고, 2.5는 이것을 24층 반복합니다. 3은 행과 열의 상호작용을 앞쪽 몇 블록에서만 하고, 24층짜리 ICL은 행 벡터 하나씩만 다룹니다.

- 열 단계(3블록): 열마다 inducing point $M$개(128)가 학습 행을 요약하고 모든 행이 그 요약을 봅니다. $F \cdot M \cdot (N + R)$
- 행 단계(3블록): 행마다 CLS $k$개와 feature 토큰 $F$개가 서로를 봅니다. $R \cdot (k + F)^2$. 마지막 블록은 CLS만 query라 $R \cdot k \cdot (k + F)$입니다(다음 노트북 2절)
- 행 ICL(24층): 행 $R$개가 학습 행 $N$개를 봅니다. $R \cdot N$. **feature 수와 상관없습니다**

attention 점수(query와 key의 내적)를 몇 번 계산하는지 세어 비교합니다. 폭(차원)과 MLP는 빼고 셉니다. 3은 셀 묶기가 토큰 수를 줄이지 않아서(3절) 토큰 열이 feature 수 $F$와 같고, 2.5는 $\lceil F/3 \rceil + 1$입니다(셀 3개 묶음 + y 토큰).

In [5]:
def scores_v25(rows, n_train, n_features, layers=24):
    c = -(-n_features // 3) + 1  # 셀 3개 묶음 + y 토큰
    feature_attention = rows * c * c
    sample_attention = c * rows * n_train
    return layers * (feature_attention + sample_attention)


def scores_v3(rows, n_train, n_features, m=128, k=4, column_blocks=3, row_blocks=3, layers=24):
    column = column_blocks * n_features * m * (n_train + rows)
    tokens = k + n_features
    # 마지막 블록은 CLS k개만 query (다음 노트북 2절)
    row = (row_blocks - 1) * rows * tokens**2 + rows * k * tokens
    icl = layers * rows * n_train
    return {"열 단계": column, "행 단계": row, "행 ICL": icl}


cost = {}
for rows, n_features in [(1_000, 10), (1_000, 100), (10_000, 100), (100_000, 100), (10_000, 1_000)]:
    n_train = int(rows * 0.8)
    v3 = scores_v3(rows, n_train, n_features)
    cost[(rows, n_features)] = {
        "2.5 (억)": scores_v25(rows, n_train, n_features) / 1e8,
        "3 (억)": sum(v3.values()) / 1e8,
        "3 / 2.5": sum(v3.values()) / scores_v25(rows, n_train, n_features),
        "3 중 행 ICL 비중": v3["행 ICL"] / sum(v3.values()),
    }
cost_table = pd.DataFrame(cost).T.rename_axis(["행 수", "feature 수"])
display(cost_table.round(3))

2.5 (억)     3 (억)  3 / 2.5  3 중 행 ICL 비중
행 수    feature 수                                            
1000   10             0.966     0.266    0.275         0.723
       100            7.014     1.104    0.157         0.174
10000  100          674.940    28.317    0.042         0.678
100000 100        67229.400  2011.168    0.030         0.955
10000  1000        6701.340   290.325    0.043         0.066

행 1,000개, feature 10개인 작은 표에서도 3의 계산이 2.5의 4분의 1쯤이고, 행이 10만 개가 되면 30분의 1까지 줄어듭니다. 2.5의 sample attention은 $C \cdot R \cdot N$이라 행과 feature가 함께 늘면 곱으로 커지는데, 3에서 $R \cdot N$을 계산하는 행 ICL은 feature 수와 상관없기 때문입니다. 대신 3의 행 단계는 $(k + F)^2$이라 feature가 아주 많은 표(1,000개)에서는 행 ICL의 비중이 7%쯤으로 떨어지고 행 단계가 계산 대부분을 차지합니다. 그래도 3블록뿐이라 24층 내내 열 방향 attention을 하는 2.5보다 훨씬 적습니다.

실제 크기 모델의 forward 시간을 CPU에서 재 봅니다. 학습 행 80%, 테스트 행 20%인 분류 표이고, 세 번 재서 가운데 값을 씁니다.

In [6]:
def forward_seconds(model, x, y_train, repeat=3):
    with torch.inference_mode():
        model(x, y_train)  # 처음 한 번은 준비(캐시)로 빼고 잰다
        seconds = []
        for _ in range(repeat):
            start = time.perf_counter()
            model(x, y_train)
            seconds.append(time.perf_counter() - start)
    return float(np.median(seconds))  # 여러 번 재서 가운데 값


torch.manual_seed(0)
timing_models = {
    "2.5 (14M)": v25.eval(),
    "3 (53M)": paper("classification", "v3").build_model().eval(),
}
timing = {}
for rows, n_features in [(1024, 10), (1024, 100), (1024, 400), (4096, 100)]:
    n_train = int(rows * 0.8)
    x_big = torch.randn(1, rows, n_features)
    y_big = torch.randint(0, 2, (1, n_train)).float()
    seconds = {name: forward_seconds(m, x_big, y_big) for name, m in timing_models.items()}
    timing[(rows, n_features)] = seconds | {
        "비율 (3 / 2.5)": seconds["3 (53M)"] / seconds["2.5 (14M)"]
    }
display(pd.DataFrame(timing).T.rename_axis(["행 수", "feature 수"]).round(2))

2.5 (14M)  3 (53M)  비율 (3 / 2.5)
행 수  feature 수                                  
1024 10              0.41     0.25          0.63
     100             2.66     0.83          0.31
     400             9.70     2.60          0.27
4096 100            16.30     3.02          0.19

파라미터가 4배 가까이 많은 3이 feature 10개에서도 더 빠르고(3분의 2~4분의 3쯤), feature 100개에서는 3분의 1, 거기서 행을 4배로 늘리면 5분의 1로 줄어듭니다. 위의 점수 개수 표와 같은 방향입니다. CPU에서 잰 값이라 실행마다 조금씩 다르고 GPU와도 다를 수 있습니다. TabPFN-3 보고서는 feature 200개면 학습 행 100만 개, feature 2,000개면 10만 개까지 검증했다고 적었는데, 이 구조가 그 바탕입니다(이 저장소의 v3 권장 크기는 그 가운데 점인 10만 행·2,000 feature, `recipes/tabpfn_v3.py`의 `SUPPORTED_SIZE`). 이제 과일 표를 넣어 세 단계를 하나씩 따라갑니다.

## 3. 셀 encoder — 셀 3개 묶기와 정답 embedding

2.5 노트북과 같은 과일 표입니다. feature 5개, 학습 행 4개(사과 2, 배 2), 정답을 가린 테스트 행 2개입니다.

In [7]:
fruit = pd.DataFrame(
    {
        "무게(g)": [210, 190, 320, 290, 200, 310],
        "당도(brix)": [13.5, 14.0, 11.0, 11.5, 13.0, 12.0],
        "지름(cm)": [7.5, 7.2, 9.0, 8.6, 7.4, 8.8],
        "산도(pH)": [3.4, 3.5, 3.9, 3.8, 3.4, 3.9],
        "껍질(mm)": [0.6, 0.5, 1.0, 0.9, 0.6, 1.0],
    }
)
y = np.array([0, 0, 1, 1])  # 학습 행의 정답: 0 = 사과, 1 = 배
n_train = 4
roles = ["학습 1", "학습 2", "학습 3", "학습 4", "테스트 5", "테스트 6"]

shown = fruit.copy()
shown["y"] = ["0 (사과)", "0 (사과)", "1 (배)", "1 (배)", "?", "?"]
shown.index = roles
display(shown)

,무게(g),당도(brix),지름(cm),산도(pH),껍질(mm),y
학습 1,210,13.5,7.5,3.4,0.6,0 (사과)
학습 2,190,14.0,7.2,3.5,0.5,0 (사과)
학습 3,320,11.0,9.0,3.9,1.0,1 (배)
학습 4,290,11.5,8.6,3.8,0.9,1 (배)
테스트 5,200,13.0,7.4,3.4,0.6,?
테스트 6,310,12.0,8.8,3.9,1.0,?


**1단계: 열마다 표준화합니다.** v2·2.5와 같습니다. 평균과 표준편차는 학습 행 4개에서만 구하고, 모델 안(`standardize_cells`)에서 계산하므로 입력에는 원래 값을 넣습니다.

In [8]:
features = fruit.to_numpy(dtype=float)
mean = features[:n_train].mean(axis=0)
std = features[:n_train].std(axis=0)  # 모델과 같이 n으로 나눈 표준편차
z = (features - mean) / std

display(pd.DataFrame(z, columns=fruit.columns, index=roles).round(2))

,무게(g),당도(brix),지름(cm),산도(pH),껍질(mm)
학습 1,-0.79,0.78,-0.77,-1.21,-0.73
학습 2,-1.16,1.18,-1.17,-0.73,-1.21
학습 3,1.25,-1.18,1.24,1.21,1.21
학습 4,0.69,-0.78,0.70,0.73,0.73
테스트 5,-0.97,0.39,-0.90,-1.21,-0.73
테스트 6,1.06,-0.39,0.97,1.21,1.21


**2단계: feature마다 토큰 하나를 만들고, 거기에 다른 열 2개를 함께 묶습니다.**
2.5는 이웃한 열 3개씩 잘라 묶음 하나로 만들고, 열 수가 3의 배수가 아니면 마지막 묶음의 빈 칸을 0으로 채웠습니다(feature 5개 → 묶음 2개, 두 번째 묶음은 [산도, 껍질, 0]).
3도 셀 3개를 토큰 하나에 넣지만 방식이 다릅니다. feature $j$의 토큰에 열 $j$, $j+1$, $j+3$의 셀을 넣고, 열 번호는 끝에서 처음으로 돌아갑니다(mod $C$). 그래서 토큰 수는 feature 수와 같고(5개), 열마다 토큰이 하나씩 생기며, 셀 하나는 토큰 3개에 들어갑니다.
열 번호에 더하는 offset은 보고서의 0, 1, 3이고, 이 저장소는 그것을 $2^j - 1$로 일반화했습니다(`group_offsets`).

여기서 헷갈리기 쉬운 점이 두 가지 있습니다.

**3에는 0으로 채우는 칸이 없습니다.** 2.5는 끝에서 열이 모자라면 0으로 채웠고, 3은 처음 열로 돌아가 채웁니다. 이렇게 바뀐 것은 묶는 단위가 달라졌기 때문입니다. 2.5는 표를 3열씩 잘라 묶음마다 토큰을 하나 만듭니다. 토큰은 $\lceil F/3 \rceil$개이고 셀 하나는 토큰 1개에만 들어갑니다. 3은 feature마다 창을 하나씩 엽니다. 토큰은 $F$개이고 셀 하나는 토큰 3개에 들어갑니다. 창을 열마다 하나씩 열면 뒤쪽 열의 창이 표 끝을 넘어가는데, 그 칸을 0 대신 처음 열의 값으로 채웁니다. 아래 표에서 '산도' 토큰의 세 번째 칸은 없는 6번째 열 대신 '당도'입니다.

**offset이 0, 1, 2가 아니라 0, 1, 3인 이유는 토큰 안의 열 사이 거리에 있습니다.** 0, 1, 2로 묶으면 토큰 하나에 든 열 3개 사이의 거리가 1, 1, 2입니다. 거리 1이 두 번 나오므로 바로 옆 열끼리는 토큰 두 개에서 겹쳐 만나고, 거리 3인 열끼리는 어느 토큰에서도 만나지 못합니다. 0, 1, 3으로 묶으면 거리가 1, 2, 3으로 모두 다릅니다. 그래서 거리 1, 2, 3인 열 쌍이 각각 토큰 하나에서 한 번씩 만나고, 같은 셀 3개로 겹침 없이 더 먼 열까지 함께 봅니다. 아래 묶음 표 다음에 2.5식 묶음을 과일 표로 만들어 보고, 열이 10개인 표에서 두 offset을 비교합니다. 과일 표는 열이 5개뿐이라 처음으로 돌아가면서 거리가 겹치므로 비교에는 쓰지 않습니다.

아래 첫 표에서 행 하나가 토큰 하나이고, 칸 1~3은 그 토큰 안에 셀이 들어가는 세 자리입니다. 칸 1에는 자기 열 $j$, 칸 2에는 $j+1$, 칸 3에는 $j+3$의 셀이 들어가고, 토큰 이름은 칸 1의 자기 열로 붙였습니다. 이 규칙은 데이터 행마다 똑같이 적용되므로 과일 표에서는 행 6개 × 토큰 5개 = 토큰 30개가 생기고, 각 토큰은 자기 행의 값 3개를 담습니다. 둘째 표는 이 규칙을 학습 1 행에 적용해 칸마다 실제로 들어가는 표준화 값을 채운 것입니다.

In [9]:
offsets = group_offsets(3)
print("offset:", offsets)

names = list(fruit.columns)
n_features = len(names)
slots = [f"칸 {k + 1} (열 j+{o})" for k, o in enumerate(offsets)]
grouping = {}  # 토큰마다 칸 3개에 들어가는 열 이름
first_row = {}  # 같은 규칙을 학습 1 행에 적용한 표준화 값
for j in range(n_features):
    columns = [(j + o) % n_features for o in offsets]
    grouping[f"토큰 {j + 1} ({names[j]})"] = [names[c] for c in columns]
    first_row[f"토큰 {j + 1} ({names[j]})"] = [f"{names[c]} {z[0, c]:.2f}" for c in columns]
display(pd.DataFrame(grouping, index=slots).T)
print("학습 1 행에 적용: 칸마다 들어가는 표준화 값")
display(pd.DataFrame(first_row, index=slots).T)

offset: (0, 1, 3)


,칸 1 (열 j+0),칸 2 (열 j+1),칸 3 (열 j+3)
토큰 1 (무게(g)),무게(g),당도(brix),산도(pH)
토큰 2 (당도(brix)),당도(brix),지름(cm),껍질(mm)
토큰 3 (지름(cm)),지름(cm),산도(pH),무게(g)
토큰 4 (산도(pH)),산도(pH),껍질(mm),당도(brix)
토큰 5 (껍질(mm)),껍질(mm),무게(g),지름(cm)


학습 1 행에 적용: 칸마다 들어가는 표준화 값


,칸 1 (열 j+0),칸 2 (열 j+1),칸 3 (열 j+3)
토큰 1 (무게(g)),무게(g) -0.79,당도(brix) 0.78,산도(pH) -1.21
토큰 2 (당도(brix)),당도(brix) 0.78,지름(cm) -0.77,껍질(mm) -0.73
토큰 3 (지름(cm)),지름(cm) -0.77,산도(pH) -1.21,무게(g) -0.79
토큰 4 (산도(pH)),산도(pH) -1.21,껍질(mm) -0.73,당도(brix) 0.78
토큰 5 (껍질(mm)),껍질(mm) -0.73,무게(g) -0.79,지름(cm) -0.77


'산도' 토큰에는 산도, 껍질, 그리고 처음으로 돌아가 '당도'가 들어갑니다. 열마다 토큰이 있으니 다음 노트북 1절의 열 단계에서 "열마다 요약한다"가 그대로 성립하고, 각 토큰은 처음부터 다른 열 두 개의 값을 함께 봅니다.

In [10]:
# 2.5식: 3열씩 잘라 묶고, 모자란 칸은 0으로 채운다
chunks = {}
for start in range(0, n_features, 3):
    members = names[start : start + 3]
    members += ["0 (채움)"] * (3 - len(members))
    chunks[f"묶음 {start // 3 + 1}"] = members
print("2.5식 묶음: 과일 표 5열을 3열씩 잘라 묶음 2개, 모자란 칸은 0")
display(pd.DataFrame(chunks, index=["칸 1", "칸 2", "칸 3"]).T)


def meeting_tokens(offsets, n_columns=10):
    # 열 0과 열 d가 함께 들어가는 토큰 수 (d = 열 거리)
    tokens = [{(j + o) % n_columns for o in offsets} for j in range(n_columns)]
    return {d: sum({0, d} <= t for t in tokens) for d in range(1, n_columns // 2 + 1)}


meetings = {
    "offset 0, 1, 2": meeting_tokens((0, 1, 2)),
    "offset 0, 1, 3 (TabPFN-3)": meeting_tokens(group_offsets(3)),
}
print("열 10개짜리 표에서 거리 d만큼 떨어진 두 열이 함께 들어가는 토큰 수")
display(pd.DataFrame(meetings).T.rename_axis(columns="열 거리"))

2.5식 묶음: 과일 표 5열을 3열씩 잘라 묶음 2개, 모자란 칸은 0


,칸 1,칸 2,칸 3
묶음 1,무게(g),당도(brix),지름(cm)
묶음 2,산도(pH),껍질(mm),0 (채움)


열 10개짜리 표에서 거리 d만큼 떨어진 두 열이 함께 들어가는 토큰 수


열 거리,1,2,3,4,5
"offset 0, 1, 2",2,1,0,0,0
"offset 0, 1, 3 (TabPFN-3)",1,1,1,0,0


2.5식 묶음은 토큰이 2개뿐이고 두 번째 묶음의 세 번째 칸은 0입니다. '지름' 열과 '산도' 열은 묶음이 달라서 한 토큰에서 만나지 못합니다. 아래 표는 열 10개짜리 표에서 거리 $d$만큼 떨어진 두 열이 몇 개의 토큰에서 함께 들어가는지 셉니다. offset 0, 1, 2는 거리 1인 쌍이 토큰 2개에서 겹치고 거리 3은 0입니다. 0, 1, 3은 거리 1, 2, 3이 모두 토큰 1개씩이라, 같은 셀 3개로 겹침 없이 더 먼 열까지 함께 봅니다. offset을 $2^j - 1$(0, 1, 3, 7, …)로 늘려도 차이가 모두 달라서 이 성질이 유지됩니다.

**3단계: 토큰마다 [값 3개, 결측 표시 3개]를 Linear 하나에 넣어 E차원 벡터로 만듭니다.** 결측 표시는 v2처럼 결측값이면 $-2$, 아니면 $0$이고, 결측 칸의 값은 0입니다. 손 계산이 실제 모델과 같은지 작은 모델로 확인합니다. 작은 `ColumnRowTransformer`를 만들고, 셀 encoder(`cell_encoder`)에 들어가는 텐서를 hook으로 꺼냅니다.

In [11]:
torch.manual_seed(0)
# 작은 설정(E 16, CLS 2개, 층 2개)에서 0으로 시작한 가중치를 채워, 입력이 출력까지 닿게 한다
tiny = randomize_zero_weights(ColumnRowTransformer(ColumnRowTransformerConfig.tiny())).eval()

x = torch.tensor(features, dtype=torch.float32)[None]  # (데이터셋 1, 행 6, feature 5)
y_train = torch.tensor(y, dtype=torch.float32)[None]  # (데이터셋 1, 학습 행 4)

captured = {}
hooks = [
    tiny.cell_encoder.register_forward_hook(
        lambda module, inputs, output: captured.update(encoder_in=inputs[0], encoder_out=output)
    ),
    tiny.column_blocks[0].register_forward_pre_hook(
        lambda module, inputs: captured.update(column_in=inputs[0])
    ),
]
with torch.no_grad():
    logits = tiny(x, y_train)
for hook in hooks:
    hook.remove()

encoder_in = captured["encoder_in"][0, 0]  # 첫 학습 행: (토큰 5, 값 3 + 결측 표시 3)
expected = np.stack([z[0, [(j + o) % n_features for o in offsets]] for j in range(n_features)])
print(
    "encoder 입력:",
    tuple(captured["encoder_in"].shape),
    " (데이터셋, 행, 토큰, 값 3 + 결측 표시 3)",
)
print("encoder 출력:", tuple(captured["encoder_out"].shape), " (데이터셋, 행, 토큰, E)")
print("값 3칸이 손 계산과 같음:", np.allclose(encoder_in[:, :3].numpy(), expected, atol=1e-5))
print("결측 표시 3칸이 모두 0:", not encoder_in[:, 3:].any().item())

encoder 입력: (1, 6, 5, 6)  (데이터셋, 행, 토큰, 값 3 + 결측 표시 3)
encoder 출력: (1, 6, 5, 16)  (데이터셋, 행, 토큰, E)
값 3칸이 손 계산과 같음: True
결측 표시 3칸이 모두 0: True


**4단계: 학습 행의 토큰에 정답 embedding을 더합니다.**
정답 y를 모델에 넣는 방법이 2.5와 다릅니다. 2.5는 y를 feature 묶음과 같은 자격의 토큰으로 만들어 행 맨 끝에 하나 더 놓았습니다. 과일 표라면 행마다 [묶음 1, 묶음 2, y 토큰]으로 토큰이 3개가 되고, 테스트 행의 y 토큰은 '빈 칸' 표시를 단 채로 들어갑니다. 토큰 축으로 이어 붙이므로 토큰이 하나 늘어납니다(`cell_transformer.py`의 `torch.cat([x_tokens, y_tokens], dim=2)`).
3은 y 토큰을 만들지 않습니다. 학습 행의 셀 토큰 5개 각각에 그 행 정답의 embedding 벡터를 같은 칸끼리 더하므로, 토큰은 5개 그대로이고 각 토큰 안에 정답이 섞여 들어갑니다. 아래 표는 행마다 토큰이 어떻게 놓이는지 보여 줍니다.

In [12]:
labels = ["사과", "사과", "배", "배", "?", "?"]

# 2.5: 묶음 토큰 2개 뒤에 y 토큰 하나를 이어 붙인다
v25_layout = pd.DataFrame(
    [["묶음 1", "묶음 2", f"y 토큰 ({'빈 칸' if label == '?' else label})"] for label in labels],
    index=roles,
    columns=["토큰 1", "토큰 2", "토큰 3"],
)
# 3: 셀 토큰 5개 그대로, 학습 행이면 각 토큰에 정답 embedding을 더한다
v3_layout = pd.DataFrame(
    [[name if label == "?" else f"{name} + e({label})" for name in names] for label in labels],
    index=roles,
    columns=[f"토큰 {j + 1}" for j in range(n_features)],
)
print("2.5: 행마다 토큰 3개 (묶음 2개 + y 토큰)")
display(v25_layout)
print("3: 행마다 토큰 5개 (e(사과)는 클래스 0의 embedding 벡터)")
display(v3_layout)

2.5: 행마다 토큰 3개 (묶음 2개 + y 토큰)


,토큰 1,토큰 2,토큰 3
학습 1,묶음 1,묶음 2,y 토큰 (사과)
학습 2,묶음 1,묶음 2,y 토큰 (사과)
학습 3,묶음 1,묶음 2,y 토큰 (배)
학습 4,묶음 1,묶음 2,y 토큰 (배)
테스트 5,묶음 1,묶음 2,y 토큰 (빈 칸)
테스트 6,묶음 1,묶음 2,y 토큰 (빈 칸)


3: 행마다 토큰 5개 (e(사과)는 클래스 0의 embedding 벡터)


,토큰 1,토큰 2,토큰 3,토큰 4,토큰 5
학습 1,무게(g) + e(사과),당도(brix) + e(사과),지름(cm) + e(사과),산도(pH) + e(사과),껍질(mm) + e(사과)
학습 2,무게(g) + e(사과),당도(brix) + e(사과),지름(cm) + e(사과),산도(pH) + e(사과),껍질(mm) + e(사과)
학습 3,무게(g) + e(배),당도(brix) + e(배),지름(cm) + e(배),산도(pH) + e(배),껍질(mm) + e(배)
학습 4,무게(g) + e(배),당도(brix) + e(배),지름(cm) + e(배),산도(pH) + e(배),껍질(mm) + e(배)
테스트 5,무게(g),당도(brix),지름(cm),산도(pH),껍질(mm)
테스트 6,무게(g),당도(brix),지름(cm),산도(pH),껍질(mm)


붙이지 않고 더하는 이유는 바로 다음 단계인 열 단계에 있습니다. 열 단계는 열마다 따로, 그 열의 토큰(행 6개)끼리만 attention합니다. y가 따로 토큰이면 y도 열 하나가 되어, '무게' 열을 요약할 때 정답을 볼 수 없습니다. 정답을 모든 셀 토큰에 더해 두면 '무게' 열 안에서도 210g이 사과 행의 값인지 배 행의 값인지 알 수 있습니다. 2.5는 층마다 행 안의 토큰끼리 보는 feature attention이 있어서 y 토큰과 feature 토큰이 서로를 볼 수 있었고, 그래서 따로 붙여도 됐습니다.

분류는 클래스마다 벡터 하나(`nn.Embedding`)이고 처음 값을 서로 직교하게 둡니다. 회귀는 `Linear(1 → E)`입니다. 테스트 행은 정답을 모르니 아무것도 더하지 않습니다.

hook으로 꺼낸 두 텐서, 곧 셀 encoder 출력과 열 단계에 들어가는 입력의 차이를 행마다 봅니다. 열 단계 입력은 (데이터셋 × 열, 행, E) 모양으로 바뀌어 있으므로 다시 맞춰 뺍니다.

In [13]:
cells_out = captured["encoder_out"][0]  # (행 6, 토큰 5, E)
column_in = (
    captured["column_in"].view(n_features, len(roles), -1).transpose(0, 1)
)  # (행 6, 토큰 5, E)
added = column_in - cells_out  # 행마다 더해진 것
class_vectors = tiny.cell_target["classification"].encode.weight.detach()  # (클래스 3, E)

added_rows = {}
for i, role in enumerate(roles):
    vector = added[i, 0]  # 첫 토큰에 더해진 것 (모든 토큰에 같다)
    row = {
        "더한 벡터의 크기": vector.norm().item(),
        "토큰끼리 같음": torch.allclose(added[i], vector.expand_as(added[i]), atol=1e-6),
    }
    for c in range(2):
        row[f"클래스 {c} 벡터와 같음"] = torch.allclose(vector, class_vectors[c], atol=1e-6)
    added_rows[role] = row
display(pd.DataFrame(added_rows).T)

gram = (class_vectors @ class_vectors.T).numpy()
display(
    pd.DataFrame(
        gram,
        index=["클래스 0", "클래스 1", "클래스 2"],
        columns=["클래스 0", "클래스 1", "클래스 2"],
    ).round(3)
)

,더한 벡터의 크기,토큰끼리 같음,클래스 0 벡터와 같음,클래스 1 벡터와 같음
학습 1,1.0,True,True,False
학습 2,1.0,True,True,False
학습 3,1.0,True,False,True
학습 4,1.0,True,False,True
테스트 5,0.0,True,False,False
테스트 6,0.0,True,False,False


,클래스 0,클래스 1,클래스 2
클래스 0,1.0,0.0,0.0
클래스 1,0.0,1.0,0.0
클래스 2,0.0,0.0,1.0


사과인 학습 1·2에는 클래스 0 벡터가, 배인 학습 3·4에는 클래스 1 벡터가 모든 토큰에 똑같이 더해졌고, 테스트 행에는 아무것도 더해지지 않았습니다. 아래 표는 클래스 벡터끼리의 내적입니다. 처음 값이 직교라 대각선 밖이 0입니다. 학습 전에는 클래스끼리 헷갈리지 않는 방향에서 시작한다는 뜻입니다.

정답 embedding은 다음 노트북 3절의 행 ICL 입력에도 한 번 더, 다른 encoder로 더합니다. 셀 단계의 정답은 열 단계와 행 단계가 "이 값이 어느 클래스 행의 값인지"를 알게 하고, 행 ICL의 정답은 행 벡터 단위의 in-context learning에 쓰입니다.

**회귀는 정답 벡터를 꺼내지 않고 계산합니다.** 클래스는 개수가 정해져 있어 클래스마다 벡터를 하나씩 둘 수 있지만, 가격 같은 연속값은 그럴 수 없습니다. 그래서 회귀의 정답 encoder는 `Linear(1 → E)`이고, 학습 행으로 표준화한 정답 $y$에서 $e(y) = y \cdot w + b$를 계산합니다. $w$와 $b$는 학습되는 E차원 벡터 두 개입니다. 같은 과일 표로 "한 개 가격이 얼마냐"를 맞히는 회귀 문제를 만들고, 손으로 따라가기 쉽게 E = 4에 $w$와 $b$를 직접 정해 계산합니다.

In [14]:
price = np.array([1500, 1300, 2800, 2500])  # 학습 행의 한 개 가격(원)
price_z = (price - price.mean()) / price.std()  # 회귀 정답은 학습 행으로 표준화한다

reg_target = TargetEncoder("regression", 1, 4)  # 회귀 정답 encoder: Linear(1 → 4)
with torch.no_grad():
    reg_target.encode.weight[:, 0] = torch.tensor([1.0, -0.5, 0.2, 0.0])  # w (손으로 정한 값)
    reg_target.encode.bias[:] = torch.tensor([0.1, 0.0, 0.0, 0.3])  # b
    e = reg_target(torch.tensor(price_z, dtype=torch.float32)[None], torch.float32)[0]  # (학습 행 4, 4)

display(
    pd.DataFrame(
        {
            "가격(원)": price,
            "y (표준화)": price_z.round(2),
            "e(y) = y·w + b": ["[" + ", ".join(f"{v:.2f}" for v in row) + "]" for row in e],
            "크기": e.norm(dim=-1).numpy().round(2),
        },
        index=roles[:n_train],
    )
)

,가격(원),y (표준화),e(y) = y·w + b,크기
학습 1,1500,-0.82,"[-0.72, 0.41, -0.16, 0.30]",0.90
학습 2,1300,-1.14,"[-1.04, 0.57, -0.23, 0.30]",1.24
학습 3,2800,1.21,"[1.31, -0.61, 0.24, 0.30]",1.50
학습 4,2500,0.74,"[0.84, -0.37, 0.15, 0.30]",0.98


가격이 비슷한 학습 1·2(1,500원·1,300원)는 벡터도 비슷하고, 비싼 학습 3·4와는 앞 두 차원의 부호가 반대입니다. 분류의 클래스 벡터는 서로 직교라 사과와 배 사이에 순서가 없지만, 회귀 벡터는 모두 $b$를 지나 $w$ 방향으로 뻗은 직선 위에 있어 값의 크기와 순서를 담습니다. $|y|$가 클수록 벡터가 길어지는 것도 다릅니다(크기 열). 이 벡터를 학습 행의 모든 셀 토큰에 더하고 테스트 행에는 더하지 않는 것은 분류와 같습니다.

## 4. 정리

| | 2.5 | 3 | 이 노트북 |
|---|---|---|---|
| 뼈대 | 셀 토큰이 24층 내내 행·열 두 방향으로 attention | 열 단계(3블록) → 행 단계(3블록) → 행 ICL(24층) | 1절 |
| 파라미터 (분류) | 1,434만 | 5,306만, 거의 다 행 ICL | 1절 |
| 층마다 계산 | $R \cdot C^2 + C \cdot R \cdot N$을 24층 | 행 ICL은 $R \cdot N$이라 feature 수와 상관없음 | 2절 |
| 셀 묶기 | 이웃한 열 3개를 잘라 묶음 하나 | 열 $j$, $j+1$, $j+3$을 feature마다 토큰 하나로 (토큰 수 = feature 수) | 3절 |
| 정답 y | 행 끝에 y 토큰을 하나 더 붙임 (토큰 수 + 1) | 학습 행의 모든 셀 토큰에 정답 embedding을 더함 (토큰 수 그대로). 분류는 클래스 벡터, 회귀는 $y \cdot w + b$ | 3절 |

파라미터는 4배 가까이 많지만 계산은 표가 클수록 오히려 적습니다. 다음 노트북 [02 열 → 행 → 행 ICL 세 단계](02_column_row_icl_stages.ipynb)에서는 이 노트북에서 만든 셀 토큰을 열 단계, 행 단계, 행 ICL, 출력에 차례로 통과시킵니다.